# 習題：數學物件、Python 型別與 AI 中的數值表示

在數學中，我們經常接觸下列概念：

- 自然數（Natural Numbers）
- 整數（Integers）
- 有理數（Rational Numbers）
- 實數（Real Numbers）
- 複數（Complex Numbers）
- 向量（Vectors）
- 矩陣（Matrices）
- 集合（Sets）
- 命題的真假（Truth Values）

本習題將探討：

```text
數學物件
    ↓
Python 型別
    ↓
CPython 實作
    ↓
AI 中的數值表示
```

---

## 第一部分：Python 如何表示數學物件？

請查閱 Python 文件或自行測試，完成下表。

| 數學物件 | Python 型別 | 範例 |
| --- | --- | --- |
| 自然數 | int | 1 |
| 整數 | int | 1 |
| 有理數 | fractions.Fraction | Fraction(3, 4) |
| 實數 | float | 1.21 |
| 複數 | complex | 2 + 3j |
| 向量 | list | [1, 2, 3] |
| 矩陣 | numpy.ndarray | np.array([[1, 2]]) |
| 集合 | set | {1, 2, 3} |
| 命題真假 | bool | True |


並回答下列問題：

1. 哪些是 Python 的內建型別（Built-in Types）？
`int`、`float`、`complex`、`list`、`set`、`bool`
2. 哪些需要額外模組才能較完整地表示？
有理數/向量 / 矩陣
3. Python 的型別分類與數學中的分類是否完全一致？
不完全一致

---

## 第二部分：數學概念與 Python 型別

數學上有下列包含關係：

$$
\mathbb N
\subseteq
\mathbb Z
\subseteq
\mathbb Q
\subseteq
\mathbb R
\subseteq
\mathbb C
$$

請回答：

1. Python 是否為上述每一個數系都提供獨立型別？
沒有
2. 為什麼自然數與整數都使用 `int` 表示？
自然數實質上只是受到限制（非負）的整數。將兩者拆分為不同型別會增加型別檢查與轉換的複雜度；統一採用 `int` 表示能保持語言設計的簡潔與一致性。
3. Python 為什麼提供 `complex` 型別？
複數在科學計算、工程學、訊號處理與物理學中非常普遍。Python 誕生初期即定位於科學與通用計算，直接內建 `complex` 語法（如 `2 + 3j`）能讓數學表達更為直覺自然。
4. Python 為什麼沒有內建的有理數型別？
有理數（分數）運算需頻繁進行約分與計算最大公因數（GCD），效能顯著低於 CPU 硬體直接支援的浮點數；絕大多數應用場景中，`float` 的精度與效能平衡已能滿足需求，故將有理數劃入標準函式庫 `fractions` 即可。
5. 下列三個物件的型別有何不同？
`1`：型別為 `int`（整數）。
`1.0`：型別為 `float`（IEEE 754 雙精度浮點數）。
`1+0j`：型別為 `complex`（複數，實部與虛部皆為 `float`）。

```python
1
1.0
1+0j
```

---

## 第三部分：CPython 如何實作這些型別？

除了知道 Python 使用哪些型別表示數學物件之外，

我們也希望了解：

> Python 物件在 CPython 中是如何實作的？

請查閱資料，完成下表。

| 數學概念 | Python 型別 | CPython 實作 | 主要特點 |
|----------|------------|-------------|----------|
| 自然數 | `int` | `PyLongObject` | 動態陣列儲存 30-bit 數字塊（Digits），無位元限制 |
| 整數 | `int` | `PyLongObject` | 動態陣列儲存 30-bit 數字塊（Digits），無位元限制 |
| 實數 | `float` | `PyFloatObject` | 包裝 C 語言的 `double`（IEEE 754 64 位元雙精度浮點數） |
| 複數 | `complex` | `PyComplexObject` | 包裝兩個 `double`（`real` 與 `imag`），共 128 位元 |
| 集合 | `set` | `PySetObject` | 雜湊表（Hash Table），開放定址法，動態擴容 |
| 命題真假 | `bool` | `PyBoolObject` | 繼承自 `PyLongObject`，全域僅有 `Py_True` 與 `Py_False` 單例 |
| 串列 | `list` | `PyListObject` | 動態指標陣列（`PyObject**`），採用 Pre-allocation 策略擴容 |

---

## 第四部分：深入探討各種型別

請針對第三部分中的每一種型別回答：

1.它是否為 Python 的內建型別？
上述 7 種型別（`int`、`float`、`complex`、`set`、`bool`、`list`）全部都是 Python 的內建型別。
2.對應的 CPython 物件結構名稱為何？
詳見第三部分表格中的 CPython 實作欄位（以 `PyObject` / `PyVarObject` 為基礎結構延伸）。
3.它如何儲存資料？
* `int`：以 30-bit（32位元系統為 15-bit）為單位的 digit 陣列進行變長儲存。
* `float`：直接儲存 C 語言原生的 `double` 浮點數值。
* `complex`：結構內包含兩個 `double` 純量（實部 `real` 與虛部 `imag`）。
* `set`：儲存 `setentry` 陣列，包含元素雜湊值（hash）與指向 key 物件的指標。
* `bool`：繼承 `int`，限制值為 `0` 或 `1`。
* `list`：儲存連續的記憶體區塊，維護指向各個 Python 物件的指標陣列。


4.是否有固定大小限制？
* `float`（64-bit）、`complex`（128-bit）、`bool`（固定 1 byte 狀態）：**有固定記憶體大小與範圍限制**。
* `int`、`set`、`list`：**沒有固定大小限制**（僅受限於系統可用記憶體與 64-bit 位址空間）。


5.若資料規模持續增加，如何擴充儲存空間？
* `int`：依數值大小自動重新分配更大空間的 digit 陣列。
* `list` 與 `set`：當元素數量達到容量閾值時，自動申請更大區塊的記憶體（通常按 1.125~1.5 倍過量分配），將舊資料移轉並釋放原空間。
---

## 第五部分：Python 整數與浮點數

執行下列程式：

```python
print(10**100)
print(10**1000)
```

回答：

1.為什麼 Python 能表示如此大的整數
因為 Python 的 `int` 採用 Arbitrary Precision（任意精度）架構，內部以動態陣列儲存數字，不受 CPU 硬體暫存器（如 64 位元）的限制。
2.Python 的 `int` 是否具有固定的位元數限制？
沒有。
3.什麼是 Arbitrary Precision（任意精度）？
指計算機能處理任意長度的數字，其精度與大小僅受限於系統可用的實體記憶體，而非硬體暫存器的固定位元寬度。
4.CPython 使用什麼物件來實作 `int`？
`PyLongObject`。
5.Python 整數大小真正的限制是什麼？
系統可用記憶體（RAM）容量大小，以及運算超長整數時的時間複雜度開銷（此外 Python 3.11+ 預設限制轉字串上限為 4300 位數）。


---

再執行：

```python
0.1 + 0.2
```

```python
0.1 + 0.2 == 0.3
```

```python
format(0.1, ".20f")
```

回答：
1.為什麼結果與數學上的答案不同？
因為電腦採用二進位 IEEE 754 格式儲存浮點數，十進位的 `0.1` 與 `0.2` 在二進位中均為**無限循環小數**，截斷儲存後產生捨入誤差（Rounding Error）。
2.浮點數是否能精確表示所有小數？
**不能**。僅能精確表示分母為 2 的冪次方的小數（例如 $0.5 = \frac{1}{2}$, $0.25 = \frac{1}{4}$）。
3.什麼是浮點數的精度（precision）？
指數字能被精確保留的有效數字位數。標準雙精度 `float`（FP64）約具備 53 位二進位尾數，相當於十進位約 15 至 17 位的有效數字。
4.什麼是浮點數的範圍（range）？
指浮點數所能表達的最大與最小數量級邊界。雙精度 `float` 範圍約在 $2.23 \times 10^{-308}$ 到 $1.79 \times 10^{308}$ 之間。
5.精度與範圍有何不同？
「範圍」決定了數值能夠有多大或多小（指數部分）；「精度」決定了數值細節能夠有多準確（尾數有效位數）。

---

## 第六部分：AI 模型中的數值表示

人工智慧模型需要儲存大量參數（Parameters）與權重（Weights）。

請查閱資料並回答：

1.AI 模型中的權重通常使用什麼型別儲存？**
低精度浮點數型別（如 FP32、FP16、BF16，或量化後的 INT8/INT4）。
2.什麼是 FP64？
Double Precision（雙精度浮點數），佔用 64 位元（1 符號位 + 11 指數位 + 52 尾數位），精度與範圍極高。
3.什麼是 FP32？
Single Precision（單精度浮點數），佔用 32 位元（1 符號位 + 8 指數位 + 23 尾數位），為傳統深度學習的標準表示法。
4.什麼是 FP16？
Half Precision（半精度浮點數），佔用 16 位元（1 符號位 + 5 指數位 + 10 尾數位），記憶體減半，但動態範圍較小，較易下溢或上溢。
5.什麼是 BF16？
Brain Floating Point 16 位元，佔用 16 位元（1 符號位 + 8 指數位 + 7 尾數位），保留與 FP32 相同的指數位數（同等動態範圍），是目前 AI 訓練的主流型別。
6.為什麼大型 AI 模型不直接使用 Python 的 `int` 作為權重？
* Python 的 `int` 是高階物件標頭，帶來極大的記憶體開銷（每個物件額外佔用 28+ bytes）。
* AI 矩陣運算需要連續且緊密的記憶體配置，以進行 GPU 平行向量化計算。
* 神經網路需要連續域的梯度更新，離散整數無法表達微小的梯度變化。


---

## 第七部分：比較不同數值表示方式

完成下表。

| 表示方式 | 位元數 | 是否有誤差 | 主要用途 |
| --- | --- | --- | --- |
| int | 變長 (32/64+) | 否 (整數無誤差) | 邏輯控制、變數計數、索引 |
| FP64 | 64 | 是 | 高精度科學計算、物理模擬、金融計算 |
| FP32 | 32 | 是 | 傳統深度學習訓練、通用 GPU 計算 |
| FP16 | 16 | 是 | 混合精度訓練、行動端推論加速 |
| BF16 | 16 | 是 | 大型語言模型（LLM）訓練與推論 |

1.哪種表示方式最精確？
整數運算為 `int`；浮點數運算中為 **FP64**。
2.哪種表示方式最節省記憶體？
**FP16** 與 **BF16**（位元數最少）。
3.哪種表示方式最常見於 AI 訓練？
**BF16*（搭配 FP32 混合精度）。
4.哪種表示方式最常見於 AI 推論（Inference）？
**FP16** 與 **BF16**（或進一步量化為 INT8 / INT4）。

---

## 第八部分：綜合思考

### 思考題一

Python 官方文件指出：

```python
int
```

具有 Unlimited Precision（無限制精度）。

請回答：

1.Unlimited Precision 是什麼意思？
指在語言邏輯層面上，整數沒有固定位元寬度的限制，只要系統有足夠記憶體，數值可以任意擴展位數。
2.它是否代表可以儲存無限大的整數？
**不能**。因為硬體設備的實體記憶體（RAM）是有限的。
3.CPython 如何實現這項特性？
將大整數拆解成多個 30-bit 的 digit 儲存於動態陣列（`PyLongObject`）中，並透過 C 語言軟體演算法實作大數加減乘除。
---

### 思考題二

比較：

```python
int
```

與

```python
float
```

回答：

1. 為什麼 `int` 幾乎沒有固定上限？
2. 為什麼 `float` 卻存在範圍限制？
3. 為什麼 `float` 可能產生誤差？

---

### 思考題三

比較下列兩種設計理念：

#### Python 的整數

```text
Arbitrary Precision
正確性優先
```

#### AI 模型常用的 FP16、BF16

```text
Limited Precision
效率優先
```

請說明：

1.兩種設計理念的差異：
* **Python `int`（Arbitrary Precision）：** 「正確性優先」。寧可犧牲計算速度與記憶體空間，也要保證結果完全精確。
* **AI 模型（FP16/BF16）：** 「效率優先」。容許微小的精度損失，以換取極致的吞吐量、傳輸頻寬與硬體計算效率。


2.各自的優點與缺點：
* **Arbitrary Precision：** 優點為精確無誤差；缺點為記憶體開銷大，無法利用 GPU 等硬體進行極速向量化運算。
* **Limited Precision（FP16/BF16）：** 優點為記憶體佔用小、吞吐量極高；缺點為具備捨入誤差，可能發生數值溢位。


3.為什麼大型語言模型通常選擇較低精度的表示方式？
神經網路本質上是高維度的統計擬合模型，對微小的數值噪聲具備極強的韌性（Robustness）。節省下來的記憶體與算力，可用於擴大模型參數與處理更多的數據，這對整體模型能力的提升遠大於維持高精度帶來的效益。
4.如果所有 AI 模型都改用 FP64，可能會產生哪些問題？
* **VRAM 爆滿：** 記憶體需求直接暴增為 BF16 的 4 倍，硬體成本急遽增加。
* **運算速度大幅下降：** 現行 GPU（如 NVIDIA Tensor Cores）針對低精度進行了硬體電路優化，FP64 的計算速度遠慢於 FP16/BF16。
* **頻寬瓶頸與能耗激增：** 數據傳輸延遲極高，整體能源消耗大幅上升。

---

### 挑戰題

請用自己的話解釋：

```python
x = 10**1000
```

為什麼 Python 可以正常運作；

而大型 AI 模型卻通常選擇使用：

```text
FP32
FP16
BF16
```

而不是任意精度整數或高精度浮點數？

請從：

- 記憶體使用量
- 計算速度
- 數值精度
- GPU 運算效率

等角度進行討論。